# 02 — Clean & Quality Check

Turn the **raw** `countries_annual` (which mixes real countries with World Bank aggregates and
has no region attached) into a clean, **country-level** interim table with region/sub-region
**and a per-observation Gini welfare-metric flag**. All logic runs as SQL inside DuckDB; output
saved to `data/interim/` as Parquet.

## What this notebook decides (and why)

**1. Drop World Bank aggregates.** The WDI response includes ~48 non-country rows — `World`,
`Euro area`, `High income`, `Sub-Saharan Africa`, etc. We separate countries from aggregates by
**joining `countries_annual.country_code` to the ISO reference on `alpha_2`**: a code that exists
in ISO 3166-1 is a real country; one that doesn't is an aggregate. (This is the approach
`config.yaml` itself flags: "filter by region != '' for countries only.")

**2. Fix two real countries the naive join would wrongly drop.** Auditing the unmatched codes
surfaced two genuine countries that fail the plain join:
- **Namibia (`NA`)** — its `alpha_2` is **NULL** in the ISO reference because the string `"NA"`
  was parsed as a missing value when the CSV loaded (the classic "Namibia NA" pandas trap).
  We patch it back to `NA`.
- **Kosovo (`XK`)** — a **user-assigned** code not in standard ISO 3166-1, so it has no reference
  row at all. We add it manually (Europe / Southern Europe).

  Both have full 1960–2025 coverage and real Gini observations, so dropping them would lose two
  legitimate countries (and ~3,300 rows).

**3. Attach region / sub-region** from the ISO reference for later grouping.

**4. Flag each Gini by welfare metric (income vs consumption).** WDI's `SI.POV.GINI` reports a
Gini but not whether it's income- or consumption-based — the single biggest cross-country
comparability hazard (income Ginis run ~4.7 pts higher on average). We attach `gini_welfare_type`
**per country × year** from the PIP source ingested in `01-ingest`, so any later ranking can label
or segment by method instead of silently mixing them. See the join cell for how ambiguous
country-years (a country reporting BOTH an income and a consumption survey the same year) are
resolved.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, quality_report, save_interim, register_source,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Step 0 — Audit: which codes match ISO, which don't

Confirm the split before we act on it: how many distinct codes match the ISO reference, and what
the unmatched ones are (should be WB aggregates **plus** the Namibia/Kosovo edge cases).

In [ ]:
summary = run_sql("""
  with codes as (select distinct country_code from countries_annual)
  select
    (select count(*) from codes) as total_codes,
    (select count(*) from codes c join countries r on c.country_code = r.alpha_2) as matched,
    (select count(*) from codes c left join countries r on c.country_code = r.alpha_2
       where r.alpha_2 is null) as unmatched
""", con)
print(summary.to_string(index=False))

print('\nUnmatched codes (48 WB aggregates + NA Namibia + XK Kosovo):')
run_sql("""
  select ca.country_code, ca.country_name
  from (select distinct country_code, country_name from countries_annual) ca
  left join countries r on ca.country_code = r.alpha_2
  where r.alpha_2 is null
  order by ca.country_code
""", con)

## Step 1 — Build a patched region crosswalk

Take the ISO reference down to just the join columns, **patch Namibia** (`alpha_3 = 'NAM'` → set
`alpha_2 = 'NA'`), and **append Kosovo** (`XK`). The result, `region_xwalk`, is the authoritative
list of real countries + their regions we'll join against.

In [ ]:
con.execute("DROP TABLE IF EXISTS region_xwalk")
con.execute("""
  CREATE TABLE region_xwalk AS
  WITH base AS (
    SELECT
      -- patch Namibia: its alpha_2 'NA' was read as NULL from the ISO CSV
      CASE WHEN alpha_3 = 'NAM' THEN 'NA' ELSE alpha_2 END AS alpha_2,
      alpha_3, name AS iso_name, region, sub_region
    FROM countries
    WHERE region IS NOT NULL          -- drop the 2 null-region ISO rows (Antarctica, Taiwan)
  )
  SELECT * FROM base
  UNION ALL
  -- Kosovo: user-assigned code XK, absent from standard ISO 3166-1
  SELECT 'XK', 'XKX', 'Kosovo', 'Europe', 'Southern Europe'
""")
print('region_xwalk rows:', con.execute('select count(*) from region_xwalk').fetchone()[0])
run_sql("select alpha_2, iso_name, region, sub_region from region_xwalk where alpha_2 in ('NA','XK')", con)

## Step 2 — Filter to countries + attach region

Inner-join `countries_annual` to `region_xwalk` on `country_code = alpha_2`. Matched rows are real
countries (aggregates fall away); we attach `alpha_3`, `region`, `sub_region`. Types are already
clean from the pivot, so this pass is a filter + enrich, not a recast. Materialized as a temp view
`_countries_geo` that Step 2b adds the welfare flag onto.

In [ ]:
con.execute("DROP VIEW IF EXISTS _countries_geo")
con.execute("""
  CREATE TEMP VIEW _countries_geo AS
  SELECT
    ca.country_code            AS iso_alpha2,
    x.alpha_3                  AS iso_alpha3,
    ca.country_name,
    x.region, x.sub_region,
    CAST(ca.year AS INTEGER)   AS year,
    ca.population, ca.gdp_per_capita_ppp, ca.gdp_per_capita_nominal, ca.gdp_growth_pct,
    ca.gini_index, ca.life_expectancy, ca.fertility_rate, ca.urban_pct,
    ca.unemployment_pct, ca.poverty_215_pct, ca.poverty_365_pct,
    ca.inflation_pct, ca.trade_pct_gdp, ca.internet_pct
  FROM countries_annual ca
  JOIN region_xwalk x ON ca.country_code = x.alpha_2
""")
n = con.execute('select count(*) from _countries_geo').fetchone()[0]
nc = con.execute('select count(distinct iso_alpha2) from _countries_geo').fetchone()[0]
print(f'_countries_geo: {n:,} rows, {nc} countries')

## Step 2b — Attach the Gini welfare metric (income vs consumption)

Join to `pip_inequality` (ingested in `01-ingest`) on **`iso_alpha3 = country_code AND year =
reporting_year`** — WDI Gini years align to PIP survey years because both derive from PIP.

**The ambiguity we resolve:** ~19% of countries (33 of 171) report BOTH an income and a
consumption survey at some point, and a handful do so in the *same* year — so a plain join would
duplicate rows and we couldn't tell which aggregate WDI actually used. Rule: for each country-year
we keep the PIP row whose `gini_pct` is **closest to the WDI `gini_index`** — that unambiguously
identifies the welfare aggregate WDI reported. After this rule the WDI-vs-PIP Gini agreement is
essentially exact (mean abs diff ≈ 0.03 pts), which we assert below.

`gini_welfare_type` is populated only where a Gini exists **and** a PIP match is found; it's NULL
otherwise (e.g. no household survey that year). We do NOT alter the Gini value itself.

In [ ]:
df_clean = run_sql("""
  WITH pip_nat AS (
    SELECT country_code, reporting_year, welfare_type, gini_pct
    FROM pip_inequality
    WHERE welfare_type IS NOT NULL AND gini_pct IS NOT NULL
  ),
  matched AS (
    SELECT g.iso_alpha3, g.year, p.welfare_type,
           ROW_NUMBER() OVER (
             PARTITION BY g.iso_alpha3, g.year
             ORDER BY abs(g.gini_index - p.gini_pct)   -- closest PIP gini = the metric WDI used
           ) AS rn
    FROM _countries_geo g
    JOIN pip_nat p ON g.iso_alpha3 = p.country_code AND g.year = p.reporting_year
    WHERE g.gini_index IS NOT NULL
  ),
  welfare AS (SELECT iso_alpha3, year, welfare_type FROM matched WHERE rn = 1)
  SELECT
    g.iso_alpha2, g.iso_alpha3, g.country_name, g.region, g.sub_region, g.year,
    g.population, g.gdp_per_capita_ppp, g.gdp_per_capita_nominal, g.gdp_growth_pct,
    g.gini_index, w.welfare_type AS gini_welfare_type,
    g.life_expectancy, g.fertility_rate, g.urban_pct, g.unemployment_pct,
    g.poverty_215_pct, g.poverty_365_pct, g.inflation_pct, g.trade_pct_gdp, g.internet_pct
  FROM _countries_geo g
  LEFT JOIN welfare w ON g.iso_alpha3 = w.iso_alpha3 AND g.year = w.year
  ORDER BY g.country_name, g.year
""", con)

gini_obs = df_clean['gini_index'].notna().sum()
flagged = df_clean['gini_welfare_type'].notna().sum()
print(f'countries_clean: {len(df_clean):,} rows, {df_clean.iso_alpha2.nunique()} countries, '
      f'years {df_clean.year.min()}-{df_clean.year.max()}')
print(f'Gini observations: {gini_obs:,}  |  with welfare_type flag: {flagged:,} '
      f'({flagged/gini_obs:.0%})')
print('welfare mix (flagged Gini obs):',
      df_clean.loc[df_clean.gini_welfare_type.notna(), 'gini_welfare_type'].value_counts().to_dict())
df_clean[df_clean.gini_index.notna()][['country_name','year','gini_index','gini_welfare_type']].head()

## Step 3 — Verify the cleaning did exactly what we intended

Assertions so a re-run fails loudly if the data shifts:
- **No aggregates leaked through** — none of the known aggregate codes survive.
- **Namibia and Kosovo are retained.**
- **Row math reconciles** — clean rows = naive-join rows + Namibia + Kosovo.
- **Welfare match is faithful** — where a flag was assigned, WDI Gini and the chosen PIP Gini
  agree to well under a point (proves we picked the right aggregate).

In [ ]:
codes = set(df_clean['iso_alpha2'])
aggregates = {'1W','1A','EU','XC','XD','OE','Z4','Z7','ZG','ZJ','XU','XM','XP','XT','XN','ZT'}
leaked = codes & aggregates
assert not leaked, f'Aggregates leaked into clean data: {leaked}'
print('✓ no WB aggregates in clean data')

for cc, nm in [('NA', 'Namibia'), ('XK', 'Kosovo')]:
    n = (df_clean['iso_alpha2'] == cc).sum()
    assert n > 0, f'{nm} ({cc}) missing from clean data!'
    print(f'✓ {nm} ({cc}) retained — {n} rows')

naive = con.execute("""select count(*) from countries_annual ca
    join countries r on ca.country_code = r.alpha_2 where r.region is not null""").fetchone()[0]
extra = con.execute("select count(*) from countries_annual where country_code in ('NA','XK')").fetchone()[0]
print(f'\nreconcile: naive-join {naive:,} + Namibia/Kosovo {extra:,} = {naive+extra:,}  vs  clean {len(df_clean):,}')
assert naive + extra == len(df_clean), 'row math does not reconcile'
print('✓ row counts reconcile')

# welfare match fidelity: WDI gini vs the PIP gini_pct we matched on, for flagged rows
fidelity = run_sql("""
  WITH pip_nat AS (SELECT country_code, reporting_year, welfare_type, gini_pct FROM pip_inequality
                   WHERE welfare_type IS NOT NULL AND gini_pct IS NOT NULL),
  m AS (SELECT g.iso_alpha3, g.year, g.gini_index, p.gini_pct,
               ROW_NUMBER() OVER (PARTITION BY g.iso_alpha3,g.year ORDER BY abs(g.gini_index-p.gini_pct)) rn
        FROM _countries_geo g JOIN pip_nat p
          ON g.iso_alpha3=p.country_code AND g.year=p.reporting_year WHERE g.gini_index IS NOT NULL)
  SELECT round(avg(abs(gini_index-gini_pct)),3) mean_abs_diff,
         round(max(abs(gini_index-gini_pct)),3) max_abs_diff FROM m WHERE rn=1
""", con)
print('\nwelfare-match fidelity (WDI vs matched PIP gini):')
print(fidelity.to_string(index=False))
assert fidelity['max_abs_diff'].iloc[0] < 1.0, 'a matched welfare_type disagrees with WDI Gini by >1 pt'
print('✓ matched welfare types agree with WDI Gini to <1 pt')

## Step 4 — Quality report

High null % on the indicator columns is **expected and correct** — WDI coverage is sparse (Gini in
particular is only measured in irregular survey years), so this is not a defect. `gini_welfare_type`
is intentionally NULL wherever there's no Gini/survey. The report is for visibility; the checks that
must hold are the assertions above. We only *require* the identity/region columns to be populated.

In [ ]:
qr = quality_report(
    df_clean, table_name='countries_clean', con=con,
    required_columns=['iso_alpha2', 'iso_alpha3', 'country_name', 'region', 'year'],
    max_null_pct=0.90,   # indicators (esp. Gini + its welfare flag) are legitimately very sparse
)
for col in ['iso_alpha2', 'iso_alpha3', 'country_name', 'region', 'sub_region', 'year']:
    assert qr['null_pcts'][col] == 0.0, f'{col} should have no nulls, got {qr["null_pcts"][col]:.1%}'
print('\n✓ identity + region columns fully populated')

## Step 5 — Persist: DuckDB table + interim Parquet + provenance

Write `countries_clean` back to DuckDB (for `03-prepare`), save the interim Parquet, and register
the source — now noting the `gini_welfare_type` flag and how it was derived.

In [ ]:
con.execute('DROP TABLE IF EXISTS countries_clean')
con.execute('CREATE TABLE countries_clean AS SELECT * FROM df_clean')

save_interim(df_clean, cfg, 'countries_clean.parquet')

register_source(
    con, 'countries_clean',
    name='Country-level WDI panel (cleaned, welfare-flagged)',
    url='https://api.worldbank.org/v2/country/all/indicator/', license='CC-BY 4.0',
    notes='countries_annual filtered to REAL COUNTRIES (ISO 3166-1 alpha-2 join), WB aggregates '
          'dropped, region/sub-region attached. Namibia (NA, NULL alpha_2 in ISO CSV) patched and '
          'Kosovo (XK) added. NEW: gini_welfare_type (income|consumption) attached per country x '
          'year from WB PIP, matched to the WDI Gini by closest gini value. NULL where no Gini/'
          'survey. Indicator columns are legitimately sparse (irregular survey coverage).',
    retrieved='2026-09-22',
    methodology='Inner join countries_annual.country_code = ISO alpha_2 (patched); region from UN '
                'geoscheme. Welfare metric from PIP joined on alpha-3 + year, disambiguated by '
                'closest-gini match to identify the aggregate WDI used.',
    series_breaks='Gini income-based vs consumption-based NOT directly comparable (income ~4.7 pts '
                  'higher). gini_welfare_type now LETS YOU SEGMENT/LABEL by metric instead of mixing. '
                  'Surveys irregular (year = nearest survey year); PPP rebasing shifts values.',
)
run_sql("select duckdb_table, source_name from _sources order by duckdb_table", con)

## Cleanup

Close the DuckDB connection (single-writer — leaving it open blocks other notebooks/scripts).

**Note for `03-prepare` / analysis:** any cross-country Gini ranking should now either filter to a
single `gini_welfare_type` or clearly label each bar by metric — the flag makes the
income-vs-consumption caveat actionable rather than just a footnote. Countries with a Gini but a
NULL flag had no matching PIP survey that year; decide per chart whether to include them (labeled
"metric unknown") or drop them.

In [ ]:
con.close()
print('Connection closed.')

---
**Next:** `03-prepare.ipynb` — feature engineering (e.g. latest-available Gini per country **with
its welfare metric**, region roll-ups) and package the export (CSV + Excel + Parquet + codebook).